In [0]:
import pandas as pd
from datetime import datetime

SCC_DIR = "/Volumes/opsanalytics_adb_workspace01/lab/raw_data/scc_data"

# --- Signatures for the ONE file we want: the daily 12:03 detail file ---
# Columns that must be PRESENT (discriminate daily-detail from stub AND weekly)
DAILY_REQUIRED = {"PATIENT MRN", "WORKSTATION", "VERIFIED_DATE", "WORK SHIFT", "CLINIC_TYPE"}
# Columns that must be ABSENT (positively exclude the weekly and stub variants)
DAILY_FORBIDDEN = {"MRN", "TESTING_WORKSTATION_ID", "VERIFIED_DT", "AGE",  # weekly-only
                   "SPEC_RECEIVED", "TECH_ID", "ORDER_DATE"}               # stub-only
MIN_DATA_ROWS = 1                # reject empty sheets; raise if you know a floor

In [0]:
def _norm(cols):
    return {str(c).strip().upper() for c in cols}

def classify_scc_file(path):
    try:
        header = pd.read_excel(path, nrows=0).columns
    except Exception as e:
        return {"kind": "unreadable", "reason": str(e), "n_cols": None, "n_rows": None}

    cols = _norm(header)
    req = {c.upper() for c in DAILY_REQUIRED}
    forb = {c.upper() for c in DAILY_FORBIDDEN}

    has_required = req.issubset(cols)
    has_forbidden = bool(forb & cols)

    if has_required and not has_forbidden:
        # passed header checks — now confirm it actually holds data
        try:
            df = pd.read_excel(path)
        except Exception as e:
            return {"kind": "unreadable", "reason": str(e), "n_cols": len(cols), "n_rows": None}

        n_rows = len(df)
        checks = {
            "rows_present": n_rows >= MIN_DATA_ROWS,
            "order_id_populated": df["ORDER_ID"].notna().any() if "ORDER_ID" in df.columns else False,
            "result_col_present": "RESULT" in _norm(df.columns),
        }
        kind = "daily_detail" if all(checks.values()) else "daily_detail_suspect"
        return {"kind": kind, "reason": None, "n_cols": len(cols), "n_rows": n_rows, "checks": checks}

    # not the daily file — label why, for the report
    if forb & cols & {"MRN", "TESTING_WORKSTATION_ID", "VERIFIED_DT", "AGE"}:
        kind = "weekly"
    elif forb & cols & {"SPEC_RECEIVED", "TECH_ID", "ORDER_DATE"}:
        kind = "stub"
    else:
        kind = "unknown"
    return {"kind": kind, "reason": None, "n_cols": len(cols), "n_rows": None}


def inventory_and_classify(scc_dir=SCC_DIR):
    results = []
    for f in dbutils.fs.ls(scc_dir):
        if f.isDir() or not f.name.lower().endswith(".xlsx"):
            continue
        info = classify_scc_file(f.path.replace("dbfs:", ""))
        info.update({
            "name": f.name,
            "path": f.path,
            "size_kb": round(f.size / 1024, 1),
            "modification_time": datetime.fromtimestamp(f.modificationTime / 1000),
        })
        results.append(info)
    return results


results = inventory_and_classify()

print(f"{'FILE':40} {'KIND':22} {'COLS':>5} {'ROWS':>8}  MODIFIED")
for r in sorted(results, key=lambda r: r["modification_time"]):
    rows = r["n_rows"] if r["n_rows"] is not None else "-"
    print(f"{r['name']:40} {r['kind']:22} {str(r['n_cols']):>5} {str(rows):>8}  {r['modification_time']}")

kept    = [r for r in results if r["kind"] == "daily_detail"]
suspect = [r for r in results if r["kind"] == "daily_detail_suspect"]
unknown = [r for r in results if r["kind"] == "unknown"]

if suspect:
    print("\n  Passed header checks but failed a value check — review before loading:")
    for r in suspect:
        failed = [k for k, v in r["checks"].items() if not v]
        print(f"   {r['name']}: failed {failed}")
if unknown:
    print(f"\n Matched no known signature: {[r['name'] for r in unknown]}")

print(f"\nSelected daily_detail ({len(kept)}): {[r['name'] for r in kept]}")
if len(kept) != 1:
    print("Expected exactly 1 daily_detail file — investigate before proceeding.")

In [0]:
NOT_DAILY_KINDS = {"stub", "weekly"}
NEEDS_REVIEW_KINDS = {"unknown", "daily_detail_suspect", "unreadable"}

def delete_non_daily(results, dry_run=True):
    to_delete = [r for r in results if r["kind"] in NOT_DAILY_KINDS]
    to_review = [r for r in results if r["kind"] in NEEDS_REVIEW_KINDS]

    print(f"Will delete {len(to_delete)} files (stub + weekly).")
    if to_review:
        print(f"  {len(to_review)} file(s) NOT deleted — need review: "
              f"{[r['name'] for r in to_review]}")

    if dry_run:
        print("Dry run only — nothing deleted. Re-run with dry_run=False to actually delete.")
        return to_delete

    deleted = []
    for r in to_delete:
        dbutils.fs.rm(r["path"])
        deleted.append(r["name"])
    print(f"Deleted {len(deleted)} files.")
    return deleted

In [0]:
delete_non_daily(results, dry_run=False)